In [116]:
from langgraph.graph import StateGraph ,START ,END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import TypedDict ,Annotated
import operator

In [117]:
load_dotenv()

True

In [118]:
model = ChatOpenAI(model='gpt-4o-mini')

In [119]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detail feedback for the essay")
    score: int = Field(description="Score out of 10",gt=0 , le =10)

In [120]:
structured_model = model.with_structured_output(EvaluationSchema)

In [121]:
essay = """
The Architecture of Anonymity: Understanding Privacy in MoneroIn the landscape of digital currencies, transparency is frequently touted as a feature. Blockchains like Bitcoin maintain public ledgers where every transaction amount, sender address, and recipient address can be viewed and tracked by anyone. While this public verification ensures trust without intermediaries, it introduces a critical flaw: a lack of financial privacy and fungibility. Monero ($XMR$) was engineered to solve this exact problem, redefining what it means to use decentralized cash by making privacy the default rather than an optional add-on.The Problem with Public LedgersOn a transparent blockchain, coins carry a history. If a specific coin was previously involved in a disputed transaction or restricted activity, it can be "tainted," leading to blacklisting or censorship by centralized exchanges and merchants. This compromises fungibility—the core economic property that all units of a currency must be completely interchangeable. Monero addresses this by ensuring that the transaction graph is cryptographically obfuscated, rendering transaction histories untraceable and individual coins indistinguishable from one another.Core Cryptographic PillarsMonero achieves absolute privacy through three primary cryptographic technologies operating in harmony:Ring Signatures: To obscure the sender, Monero bundles the true spender's signature with a group of decoys drawn from the blockchain. An outside observer cannot determine which member of the ring actually authorized the transfer.Stealth Addresses: To protect the recipient, the sender generates a unique, one-time destination address for every single transaction. Even if multiple payments are sent to the same person, each transaction goes to a completely distinct address on-chain, preventing linkability.Ring Confidential Transactions (RingCT): Transparency of amounts is eliminated using cryptographic commitments. RingCT hides the exact quantity of currency transferred while mathematically proving to the network that no coins were created out of thin air and that the inputs match the outputs.The Broader Implications of Financial PrivacyCritics often associate privacy coins exclusively with illicit activities, but financial privacy is fundamentally a prerequisite for basic human autonomy and commerce. Just as physical cash allows individuals to buy groceries or donate to causes without broadcasting their bank statements to the world, digital cash requires similar protections. Without privacy, automated profiling, economic surveillance, and censorship become structural features of the financial system.ConclusionMonero represents a profound philosophical and technical shift in cryptocurrency design. By embedding privacy directly into the protocol layer, it ensures that financial sovereignty remains accessible to everyone. In an era where data collection is ubiquitous, Monero stands as a technical testament to the idea that privacy is not a crime, but a fundamental human right.
"""

essay2 ="""
Monero is like a coin that hides everything u do unlike bitcoin where cops or anyone can see ur wallet and how much money u got. monero make all transaction private by default which is super cool for privacy nerds.

basically it use ring signatures to fake the sender so nobody know who send what, and stealth addresses so the receiver address is always random each time, plus ringct hide the actual amount of money moving around.

some people say it is only for bad stuff like dark web and crime, but honestly privacy is a human right bro. if banks and gov can track every single rupee u spend thats scary and bad for freedom. monero fix this by making sure nobody can spy on ur bags.
"""

In [122]:
prompt = f'Evaluate the langugage quality of essay and provice a feedback and assign a score out of 1o \n {essay}'
structured_model.invoke(prompt)


EvaluationSchema(feedback="The essay presents a well-structured argument advocating for the importance of privacy in cryptocurrency with a focus on Monero. The language is formal and appropriate for an academic audience, utilizing technical terminology effectively. However, there are some areas for improvement in sentence complexity and clarity. The essay would benefit from simpler sentence structures in certain sections to improve readability for a wider audience. Additionally, while the essay maintains a strong focus on the topic, a more engaging introduction and conclusion could enhance its overall impact. Some phrases could be rephrased for conciseness or clarity, and additional examples or analogies could further clarify complex concepts for readers unfamiliar with cryptocurrency. Overall, it's a strong piece with minor areas that could be improved for clarity and engagement.", score=8)

In [123]:
class UPSCState(TypedDict):
    essay:str
    language_feedback :str
    analysis_feedback :str
    clarity_feedback :str
    overall_feedback :str
    individual_scores: Annotated[list[int],operator.add]
    avg_score :float

In [124]:
def evaluate_language(state :UPSCState):
    prompt = f'Evaluate the langugage quality of essay and provice a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)
    return {'language_feedback':output.feedback ,'individual_scores':[output.score]}


In [125]:

def evaluate_analysis(state :UPSCState):
    prompt = f'Evaluate the depth of analysis  of essay and provice a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)
    return {'analysis_feedback':output.feedback ,'individual_scores':[output.score]}

In [126]:


def evaluate_thought(state :UPSCState):
    prompt = f'Evaluate the clarity of tought of of essay and provice a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)
    return {'clarity_feedback':output.feedback ,'individual_scores':[output.score]}

In [127]:


def final_evaluation(state :UPSCState):

    #summary feedback
    prompt = f'Based on the follwing feedback . create a summarized feedback \n langugage_feedback = {state["language_feedback"]} \n deepth_feedback = {state['analysis_feedback'] } , \n clarity of feedback = {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt)
    #avg calculation
    avg_score = sum(state['individual_scores'])/ len(state['individual_scores'])
    return {'overall_feedback':overall_feedback ,'avg_score':avg_score}

In [128]:
graph = StateGraph(UPSCState)

graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)


In [129]:
# add edges
graph.add_edge(START , "evaluate_language")
graph.add_edge(START , "evaluate_analysis")
graph.add_edge(START , "evaluate_thought")

graph.add_edge("evaluate_language" ,"final_evaluation")
graph.add_edge( "evaluate_analysis","final_evaluation")
graph.add_edge( "evaluate_thought","final_evaluation")

graph.add_edge( "final_evaluation",END)

workflow = graph.compile()


In [ ]:
intial_state = {
"essay":essay2
}
workflow.invoke(intial_state)